# Setup packages

In [1]:
import omicstl
import pandas as pd
import numpy as np
from pathlib import Path

# Bring in Data


In [2]:
from pathlib import Path

repo_root = Path("/workspaces/timed-hpc")
data_dir = repo_root / "viral_use_case" / "data"
out_dir = repo_root / "viral_use_case" / "model_outputs"
out_dir.mkdir(parents=True, exist_ok=True)

source_path = data_dir / "source_dset.csv"
target_transfer_path = data_dir / "target_transfer.csv"
target_validation_path = data_dir / "target_validation.csv"

print("Using files:")
print(source_path)
print(target_transfer_path)
print(target_validation_path)

Using files:
/workspaces/timed-hpc/viral_use_case/data/source_dset.csv
/workspaces/timed-hpc/viral_use_case/data/target_transfer.csv
/workspaces/timed-hpc/viral_use_case/data/target_validation.csv


In [3]:
# Read data
# =========================================
base_source_data = pd.read_csv(source_path).set_index("SampleID")
base_target_transfer_data = pd.read_csv(target_transfer_path).set_index("SampleID")
base_target_validation_data = pd.read_csv(target_validation_path).set_index("SampleID")

for name, df in {
    "source": base_source_data,
    "target_transfer": base_target_transfer_data,
    "target_validation": base_target_validation_data,
}.items():
    if "Resp" not in df.columns:
        raise ValueError(f"{name} is missing 'Resp'")
    df["Resp"] = df["Resp"].apply(lambda x: 2 if x == "viral" else 1)

print("\nShapes:")
print("source:", base_source_data.shape)
print("target_transfer:", base_target_transfer_data.shape)
print("target_validation:", base_target_validation_data.shape)

print("\nResponse counts:")
print(base_source_data["Resp"].value_counts(dropna=False))
print(base_target_transfer_data["Resp"].value_counts(dropna=False))
print(base_target_validation_data["Resp"].value_counts(dropna=False))


Shapes:
source: (358, 476)
target_transfer: (192, 476)
target_validation: (48, 476)

Response counts:
Resp
1    179
2    179
Name: count, dtype: int64
Resp
2    147
1     45
Name: count, dtype: int64
Resp
2    33
1    15
Name: count, dtype: int64


# Setup Data Container

In [4]:
# =========================================
#  Dataset container
# =========================================
from omicstl.simulation_utils.data_utils import DatasetContainer
all_datasets = DatasetContainer(
    source_data=base_source_data,
    target_data=base_target_transfer_data,
    target_test_data=[base_target_validation_data],
)
all_datasets.set_response_column("Resp")

source_input = all_datasets.source_data.drop(columns=["Resp"]).copy()
target_input = all_datasets.target_test_data[0].drop(columns=["Resp"]).copy()
target_truth = all_datasets.target_test_data[0]["Resp"].copy()

feature_names = source_input.columns.tolist()

print("\nFeature matrix shapes:")
print("source_input:", source_input.shape)
print("target_input:", target_input.shape)



Feature matrix shapes:
source_input: (358, 475)
target_input: (48, 475)


# Model Fitting

In [5]:
# 4. Params
# =========================================
param_grid = {
    "dropout": [0.25, 0.5],
    "n_latent_dims": [2],
    "hidden_dim_base": [6],
    "lr": [0.01, 0.001],
    "source_epochs": [1000],
    "target_epochs": [1000],
    "freeze": ["none"],
    "weight_decay": [1e-4, 1e-2],
    "gamma": [1, 2, 3],
}


In [6]:
# =========================================
#  Fit models
# =========================================
import torch
from torch import device
import random
from omicstl.simulation_utils.model_utils import fit_dl_model, fit_rf_model
random.seed(1123)
torch.manual_seed(42)

mlp_out = mlp_model = mlp_model_targetonly = None
vae_out = vae_model = vae_model_targetonly = None
rf_out = rf_model = None

try:
    mlp_out, mlp_model, mlp_model_targetonly = fit_dl_model(
        all_datasets,
        "mult_mlp",
        device("cpu"),
        param_grid,
    )
    print("\nMLP fit complete")
except Exception as e:
    print("\nFailed MLP")
    print(e)

torch.manual_seed(42)
try:
    vae_out, vae_model, vae_model_targetonly = fit_dl_model(
        all_datasets,
        "mult_vae",
        device("cpu"),
        param_grid,
    )
    print("\nVAE fit complete")
except Exception as e:
    print("\nFailed VAE")
    print(e)

random.seed(42)
try:
    rf_out, rf_model = fit_rf_model(all_datasets)
    print("\nRF fit complete")
except Exception as e:
    print("\nFailed RF")
    print(e)

/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)



MLP fit complete


/workspaces/timed-hpc/src/omicstl/simulation_utils/model_utils.py:1039: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)



VAE fit complete

Failed RF
unsupported operand type(s) for -: 'list' and 'int'


In [9]:
# =========================================
# 6. Helper: create partition
# =========================================
def build_partition(X_features: pd.DataFrame):
    X_copy = X_features.copy()
    X_copy["Resp"] = 1
    dpart = create_data_partition(
        data=X_copy,
        response_id="Resp",
        feature_cols=feature_names,
    )
    return dpart


target_partition = build_partition(target_input)

In [ ]:
# =========================================
#  Predictions: MLP
# =========================================
mlp_pred_df = None

if mlp_model is not None:
    mlp_probs = mlp_model.__getattribute__("target").predict(
        [target_partition.features],
        return_probabilities=True,
    )
    mlp_probs = np.asarray(mlp_probs)

    if mlp_probs.ndim == 1:
        mlp_probs = mlp_probs.reshape(-1, 1)

    mlp_pred_df = pd.DataFrame(
        mlp_probs,
        index=target_input.index,
        columns=[f"mlp_prob_class_{i+1}" for i in range(mlp_probs.shape[1])]
    )
    mlp_pred_df["mlp_pred_class"] = np.argmax(mlp_probs, axis=1) + 1

In [11]:
# =========================================
#  Predictions: VAE
# =========================================
vae_pred_df = None

if vae_model is not None:
    vae_probs = vae_model.__getattribute__("target").predict(
        [target_partition.features],
        return_probabilities=True,
    )
    vae_probs = np.asarray(vae_probs)

    if vae_probs.ndim == 1:
        vae_probs = vae_probs.reshape(-1, 1)

    vae_pred_df = pd.DataFrame(
        vae_probs,
        index=target_input.index,
        columns=[f"vae_prob_class_{i+1}" for i in range(vae_probs.shape[1])]
    )
    vae_pred_df["vae_pred_class"] = np.argmax(vae_probs, axis=1) + 1


In [ ]:
# =========================================
#  Predictions: RF
# =========================================
rf_pred_df = None

def get_rf_prediction_df(model, dpart):
    all_preds = model.generate_predictions(
        views=[dpart.features],
        response=dpart.response,
        validation_views=[dpart.features],
        validation_response=dpart.response,
        ensemble_views=None,
        ensemble_response=None,
        integration_type=TransferForest.IntegrationType.NONE,
    )

    all_preds = all_preds[0]

    groupings = {
        "pred_ensemble_full": [r"^pred_ensemble(?!.*val).*"],
    }

    all_preds_reorg = {}
    for new_key, patterns in groupings.items():
        matched_keys = []
        for pattern in patterns:
            matched_keys.extend([key for key in all_preds.keys() if re.match(pattern, key)])
        if matched_keys:
            all_preds_reorg[new_key] = pd.DataFrame(
                {key: all_preds[key] for key in matched_keys}
            )

    final_preds_df = all_preds_reorg.get("pred_ensemble_full")
    if final_preds_df is None:
        raise ValueError("Could not find pred_ensemble_full in RF predictions")

    return final_preds_df

if rf_model is not None:
    rf_raw_pred_df = get_rf_prediction_df(rf_model, target_partition).copy()
    rf_raw_pred_df.index = target_input.index

    prob_cols = [c for c in rf_raw_pred_df.columns if c != "pred_ensemble"]

    rf_pred_df = pd.DataFrame(index=target_input.index)

    for i, col in enumerate(prob_cols, start=1):
        rf_pred_df[f"rf_prob_class_{i}"] = rf_raw_pred_df[col].values

    if prob_cols:
        rf_pred_df["rf_pred_class"] = np.argmax(
            rf_pred_df[[f"rf_prob_class_{i}" for i in range(1, len(prob_cols) + 1)]].values,
            axis=1,
        ) + 1

    if "pred_ensemble" in rf_raw_pred_df.columns:
        rf_pred_df["rf_pred_ensemble_raw"] = rf_raw_pred_df["pred_ensemble"].values

In [ ]:
# =========================================
# Combine and save predictions
# =========================================
predictions_df = pd.DataFrame(index=target_input.index)
predictions_df.index.name = "SampleID"
predictions_df["true_class"] = target_truth

if mlp_pred_df is not None:
    predictions_df = predictions_df.join(mlp_pred_df)

if vae_pred_df is not None:
    predictions_df = predictions_df.join(vae_pred_df)

if rf_pred_df is not None:
    predictions_df = predictions_df.join(rf_pred_df)

predictions_path = out_dir / "model_predictions.csv"
predictions_df.reset_index().to_csv(predictions_path, index=False)

print(f"\nSaved predictions to: {predictions_path}")

print("\nPredictions preview:")
display(predictions_df.head(20))



Saved predictions to: /workspaces/timed-hpc/viral_use_case/model_outputs/model_predictions.csv

Predictions preview:


,true_class,mlp_prob_class_1,mlp_prob_class_2,mlp_pred_class,vae_prob_class_1,vae_prob_class_2,vae_pred_class,rf_prob_class_1,rf_prob_class_2,rf_pred_class,rf_pred_ensemble_raw
SampleID,,,,,,,,,,,
Uni_MCK_D3_12_Hr_R5,1,0.268941,0.731059,2,0.449401,0.550599,2,0.059960,0.940040,2,2
Uni_CoV2_IT_D1_12_Hr_R3,2,0.268941,0.731059,2,0.449401,0.550599,2,0.053632,0.946368,2,2
Uni_NL63_D3_12_Hr_R3,2,0.268941,0.731059,2,0.449401,0.550599,2,0.000217,0.999783,2,2
Uni_MCK_D3_12_Hr_R2,1,0.268941,0.731059,2,0.449401,0.550599,2,0.000744,0.999256,2,2
Uni_NL63_D3_12_Hr_R5,2,0.268941,0.731059,2,0.449401,0.550599,2,0.000427,0.999573,2,2
Uni_CoV2_IT_D1_12_Hr_R4,2,0.268941,0.731059,2,0.449401,0.550599,2,0.236404,0.763596,2,2
Uni_CoV2_IT_D3_12_Hr_R4,2,0.268941,0.731059,2,0.449401,0.550599,2,0.018688,0.981312,2,2
Uni_CoV2_IT_D2_12_Hr_R2,2,0.268941,0.731059,2,0.449401,0.550599,2,0.010626,0.989374,2,2
Uni_NL63_D2_12_Hr_R5,2,0.268941,0.731059,2,0.449401,0.550599,2,0.001638,0.998362,2,2


In [15]:
# =========================================
# 11. Simple accuracy summary
# =========================================
metrics = {}

if "mlp_pred_class" in predictions_df.columns:
    metrics["mlp_accuracy"] = (predictions_df["mlp_pred_class"] == predictions_df["true_class"]).mean()

if "vae_pred_class" in predictions_df.columns:
    metrics["vae_accuracy"] = (predictions_df["vae_pred_class"] == predictions_df["true_class"]).mean()

if "rf_pred_class" in predictions_df.columns:
    metrics["rf_accuracy"] = (predictions_df["rf_pred_class"] == predictions_df["true_class"]).mean()

metrics_df = pd.DataFrame({"metric": list(metrics.keys()), "value": list(metrics.values())})
metrics_path = out_dir / "model_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print(f"\nSaved metrics to: {metrics_path}")
display(metrics_df)


Saved metrics to: /workspaces/timed-hpc/viral_use_case/model_outputs/model_metrics.csv


,metric,value
0,mlp_accuracy,0.687500
1,vae_accuracy,0.687500
2,rf_accuracy,0.770833


In [16]:
# =========================================
# 12. SHAP prediction wrappers
# =========================================
def _build_partition_from_features(X, feature_names):
    X_copy = X.copy()

    if isinstance(X_copy, np.ndarray):
        X_copy = pd.DataFrame(X_copy, columns=feature_names)

    X_copy = X_copy.loc[:, feature_names].copy()
    X_copy["Resp"] = 1

    dpart = create_data_partition(
        data=X_copy,
        response_id="Resp",
        feature_cols=feature_names,
    )
    return dpart


def mlp_shap_pred(X):
    dpart = _build_partition_from_features(X, feature_names)
    preds = mlp_model.__getattribute__("target").predict(
        [dpart.features],
        return_probabilities=True,
    )
    return preds


def vae_shap_pred(X):
    dpart = _build_partition_from_features(X, feature_names)
    preds = vae_model.__getattribute__("target").predict(
        [dpart.features],
        return_probabilities=True,
    )
    return preds


def rf_shap_pred(X):
    dpart = _build_partition_from_features(X, feature_names)

    all_preds = rf_model.generate_predictions(
        views=[dpart.features],
        response=dpart.response,
        validation_views=[dpart.features],
        validation_response=dpart.response,
        ensemble_views=None,
        ensemble_response=None,
        integration_type=TransferForest.IntegrationType.NONE,
    )

    all_preds = all_preds[0]

    groupings = {
        "pred_ensemble_full": [r"^pred_ensemble(?!.*val).*"],
    }

    all_preds_reorg = {}
    for new_key, patterns in groupings.items():
        matched_keys = []
        for pattern in patterns:
            matched_keys.extend([key for key in all_preds.keys() if re.match(pattern, key)])
        if matched_keys:
            all_preds_reorg[new_key] = pd.DataFrame(
                {key: all_preds[key] for key in matched_keys}
            )

    final_preds_df = all_preds_reorg.get("pred_ensemble_full")
    if final_preds_df is None:
        raise ValueError("Could not find pred_ensemble_full in rf predictions")

    predicted_probs = final_preds_df.drop(columns="pred_ensemble", errors="ignore")
    return predicted_probs.to_numpy()


In [ ]:
# =========================================
# 13. SHAP helpers
# =========================================
background_size = min(50, source_input.shape[0])
eval_size = min(50, target_input.shape[0])

background = source_input.sample(background_size, random_state=1)
target_eval = target_input.iloc[:eval_size].copy()

print("\nSHAP background shape:", background.shape)
print("SHAP eval shape:", target_eval.shape)


def get_class_matrix(shap_values, class_index=1):
    if isinstance(shap_values, list):
        return np.array(shap_values[class_index])

    vals = np.array(shap_values)
    if vals.ndim == 3:
        return vals[:, :, class_index]

    raise ValueError(f"Unexpected SHAP structure with shape {vals.shape}")


def save_shap_outputs(shap_matrix, X_df, prefix, class_label="class1"):
    shap_df = pd.DataFrame(shap_matrix, columns=X_df.columns)
    shap_csv = out_dir / f"{prefix}_{class_label}_shap_values.csv"
    shap_df.to_csv(shap_csv, index=False)

    plt.figure()
    shap.summary_plot(shap_matrix, X_df, show=False)
    plt.tight_layout()
    plt.savefig(out_dir / f"{prefix}_{class_label}_summary.png", dpi=300, bbox_inches="tight")
    plt.close()

    plt.figure()
    shap.summary_plot(shap_matrix, X_df, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(out_dir / f"{prefix}_{class_label}_bar.png", dpi=300, bbox_inches="tight")
    plt.close()

    return shap_df



SHAP background shape: (10, 475)
SHAP eval shape: (10, 475)


In [18]:

# =========================================
# 14. MLP SHAP
# =========================================
if mlp_model is not None:
    try:
        random.seed(1)
        torch.manual_seed(1)

        mlp_explainer = shap.KernelExplainer(mlp_shap_pred, background)
        mlp_shap_values = mlp_explainer.shap_values(target_eval, nsamples=10)

        mlp_class1 = get_class_matrix(mlp_shap_values, class_index=1)
        mlp_shap_df = save_shap_outputs(mlp_class1, target_eval, prefix="mlp", class_label="class1")

        print("\nMLP SHAP preview:")
        display(mlp_shap_df.head())

    except Exception as e:
        print("\nFailed MLP SHAP")
        print(e)


# =========================================
# 15. VAE SHAP
# =========================================
if vae_model is not None:
    try:
        random.seed(1)
        torch.manual_seed(1)

        vae_explainer = shap.KernelExplainer(vae_shap_pred, background)
        vae_shap_values = vae_explainer.shap_values(target_eval, nsamples=10)

        vae_class1 = get_class_matrix(vae_shap_values, class_index=1)
        vae_shap_df = save_shap_outputs(vae_class1, target_eval, prefix="vae", class_label="class1")

        print("\nVAE SHAP preview:")
        display(vae_shap_df.head())

    except Exception as e:
        print("\nFailed VAE SHAP")
        print(e)


# =========================================
# 16. RF SHAP
# =========================================
if rf_model is not None:
    try:
        random.seed(1)
        torch.manual_seed(1)

        rf_explainer = shap.KernelExplainer(rf_shap_pred, background)
        rf_shap_values = rf_explainer.shap_values(target_eval, nsamples=10)

        rf_class1 = get_class_matrix(rf_shap_values, class_index=1)
        rf_shap_df = save_shap_outputs(rf_class1, target_eval, prefix="rf", class_label="class1")

        print("\nRF SHAP preview:")
        display(rf_shap_df.head())

    except Exception as e:
        print("\nFailed RF SHAP")
        print(e)


print(f"\nAll saved outputs are in: {out_dir}")

  0%|          | 0/10 [00:00<?, ?it/s]


MLP SHAP preview:


,RS30_HUMAN,EIF3I_HUMAN,RL24_HUMAN,ARPC2_HUMAN,RUVB1_HUMAN,TEBP_HUMAN,PPGB_HUMAN,KPYM_HUMAN,PRDX5_HUMAN,PDIA1_HUMAN,...,RL30_HUMAN,TMM43_HUMAN,SMD3_HUMAN,VDAC2_HUMAN,RL27A_HUMAN,RAB5C_HUMAN,CPNE1_HUMAN,SRSF3_HUMAN,FKB1A_HUMAN,SC61B_HUMAN
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


  0%|          | 0/10 [00:00<?, ?it/s]


VAE SHAP preview:


...

  0%|          | 0/10 [00:00<?, ?it/s]

/opt/venv/lib/python3.12/site-packages/shap/explainers/_kernel.py:765: UserWarning: Linear regression equation is singular, a least squares solutions is used instead.
To avoid this situation and get a regular matrix do one of the following:
1) turn up the number of samples,
2) turn up the L1 regularization with num_features(N) where N is less than the number of samples,
3) group features together to reduce the number of inputs that need to be explained.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 2 iterations, i.e. alpha=4.262e+00, with an active set of 2 regressors, and the smallest cholesky pivot element being 2.220e-16. Reduce max_iter or increase eps parameters.
  warnings.warn(
/opt/venv/lib/python3.12/site-packages/shap/explainers/_kernel.py:765: UserWarning: Linear regression equation is singular, a least squares solutions is used instead.
To


RF SHAP preview:


,RS30_HUMAN,EIF3I_HUMAN,RL24_HUMAN,ARPC2_HUMAN,RUVB1_HUMAN,TEBP_HUMAN,PPGB_HUMAN,KPYM_HUMAN,PRDX5_HUMAN,PDIA1_HUMAN,...,RL30_HUMAN,TMM43_HUMAN,SMD3_HUMAN,VDAC2_HUMAN,RL27A_HUMAN,RAB5C_HUMAN,CPNE1_HUMAN,SRSF3_HUMAN,FKB1A_HUMAN,SC61B_HUMAN
0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.078489,0.730135,0.029036,0.029036,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.083072,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,-0.016392,0.0,0.000000,0.000000,-0.173988,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.777016,0.0
3,0.0,0.0,0.0,0.744215,0.0,0.475747,0.000000,0.000000,0.000000,0.020833,...,0.0,-0.718704,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
4,0.0,0.0,0.0,0.039189,0.0,0.000000,0.000000,0.546009,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0



All saved outputs are in: /workspaces/timed-hpc/viral_use_case/model_outputs
